In [ ]:
from vistiq.io import ImageWriterConfig, ImageWriter, ImageLoader, ImageLoaderConfig, unstack_image
from vistiq.utils import ArrayIteratorConfig, check_device, resolve_futures 
from vistiq.core import Tiler, TilerConfig, Untiler, UntilerConfig
from vistiq.preprocess import FuncProcessor, FuncProcessorConfig, PreprocessFlow, PreprocessFlowConfig, ResizeConfig, Resize, RescaleConfig, Rescale, DoG, DoGConfig, PreprocessorConfig, Preprocessor
from vistiq.segment import RegionFilterConfig, RegionFilter, RangeFilterConfig, RangeFilter, RegionAnalyzerConfig, RegionAnalyzer 
from vistiq.segment import MicroSAMSegmenter, MicroSAMSegmenterConfig, MicroSAMMerger, MicroSAMMergerConfig
from vistiq.segment import TiledSegmentationFlow, TiledSegmentationFlowConfig, SegmentationFlow, SegmentationFlowConfig
from vistiq.analysis import CoincidenceDetectorConfig, CoincidenceDetector, AnalysisFlowConfig, AnalysisFlow, IoSMetricsCalculatorConfig
from vistiq.core import labels_to_masks
from vistiq.analysis.overlap import (
    OverlapCalculator,
    LabelOverlapCalculatorConfig,   # or BoxOverlapCalculatorConfig / MaskOverlapCalculatorConfig
    IoSMetricsCalculatorConfig,
    metrics_calculator_configs,
    region_map_from_dataframe,      # if using region maps from DataFrames
)
from vistiq.analysis import MatrixAggregator, MatrixAggregatorConfig, MatrixCombiner, MatrixCombinerConfig
from vistiq.constant.matrix import UPPER
from vistiq.constant import LOWER_ND
from vistiq.segment.select import ValueFilter, ValueFilterConfig, TopKFilter, TopKFilterConfig
from vistiq.graph import NXGraphBuilder, NXGraphBuilderConfig, NXGraphQuery, NXGraphQueryConfig

from prefect import flow, task
from prefect.task_runners import ProcessPoolTaskRunner
from prefect.futures import wait
from prefect.futures import resolve_futures_to_results

import stackview
import os
import copy
import numpy as np
import math
import logging
import pandas as pd
import itertools

from typing import Any, List, Tuple
from pathlib import Path

# Configure logger and check availability of accelerators

In [ ]:
import vistiq
logger = logging.getLogger(vistiq.__name__)

logger.info(f"Available Torch accelerators: {check_device()}")

# Load image

In [ ]:
#path="/standard/vol191/siegristlab/Microsam_Segmentation/Conditional Split/control_24+48/DCP1/1_Dpn.tif"
#path="/standard/vol191/siegristlab/Microsam_Segmentation/24h/AkhGal4 x OR Susie/Scrib488 Dpn555 EdU 647/Raw files/Animal 1.lif"
path="Animal 1.lif"
#path="/Users/khs3z/Documents/SDS_/projects/Siegrist/Microsam_Segmentation/24h/AkhGal4 x OR Susie/Scrib488 Dpn555 EdU 647/Animal 1.lif"

scene_index = 0

embedding_path = "./embeddings"
#embedding_path = "/standard/vol191/siegristlab/Sagar/microsam/embeddings/"

In [ ]:
ilc = ImageLoaderConfig(
    squeeze=True, 
    rename_channel={"Red": "Dpn", "Green": "Scrib", "Blue": "EdU"}, 
    scene_index=scene_index, 
    split_channels=False,
    substack="Z:20-50"
)
img, metadata = ImageLoader(ilc).run(path)

In [ ]:
if "C" in metadata["axes"]:
    vimg = np.concatenate(np.unstack(img, axis=0), axis=-1)
else:
    vimg = img
stackview.slice(vimg)
#stackview.switch(img, colormap=["pure_green", "pure_blue", "pure_red"], toggleable=True)

# Preprocess

In [ ]:
tissue_ppcfg = PreprocessFlowConfig(
    processors = [
        RescaleConfig(
            low=2, 
            high=98, 
            dtype=np.uint8, 
            iterator_config=ArrayIteratorConfig(slice_def=(-3,-2,-1)) # ZYX over each channel
        ),
        FuncProcessorConfig(
            func="skimage.filters.gaussian",
            kwargs={"sigma": 1.0},
            iterator_config=ArrayIteratorConfig(slice_def=(-2,-1)) # YX over each Z-plane and channel
        ),
        FuncProcessorConfig(
            func="skimage.exposure.adjust_gamma",
            kwargs={"gamma": 0.2},
            iterator_config=ArrayIteratorConfig(slice_def=(-3,-2,-1)) # ZYX over each channel
        ),
        FuncProcessorConfig(
            func="skimage.exposure.adjust_sigmoid",
            iterator_config=ArrayIteratorConfig(slice_def=(-3,-2,-1)) # ZYX over each channel
        ),
        RescaleConfig(
            dtype=np.uint8, 
            iterator_config=ArrayIteratorConfig(slice_def=(-3,-2,-1)) # ZYX over each channel
        ),
        FuncProcessorConfig(
            func="numpy.max", 
            kwargs={"axis":("C")}, # Project all channels into one
            strict_axis=False,     # don't throw exception if the input is a single channel image already
            dtype=np.uint16,
        ),
    ]
)
#c_img, c_metadata = PreprocessFlow(tissue_ppcfg).run(img, metadata=metadata, workers=-1)
#metadata, c_metadata

In [ ]:
# stackview.slice(c_img)

# Configuration for 3D Tissue Segmentation

In [ ]:
mscfg = MicroSAMSegmenterConfig(
    iterator_config=ArrayIteratorConfig(slice_def=()),
    embedding_path=embedding_path,
)

rfcfg = RegionFilterConfig(
    filters=[
        RangeFilterConfig(
            attribute="cross_sectional_area-xy", 
            range=(2000, np.inf)
        ),
        RangeFilterConfig(
            attribute="cross_sectional_area-xz", 
            range=(2000, np.inf)
        ),
        RangeFilterConfig(
            attribute="cross_sectional_area-yz", 
            range=(2000, np.inf)
        ),
        RangeFilterConfig(
            attribute="aspect_ratio", 
            range=(0.5, 1.0)
        ),
    ]
)

tsfcfg = TiledSegmentationFlowConfig(
    segmenter = mscfg,
    region_filter = rfcfg,
    tile_factor=(3,3),
    resize_factor=(0.25, 0.25),
    iou_threshold=0.5,
    consensus_threshold=0.75,
)

# Configuration for Region Analysis

In [ ]:
racfg = RegionAnalyzerConfig(
    properties=["slice_annotations","volume", "centroid", "cross_sectional_area", "bbox", "aspect_ratio"],
    iterator_config = ArrayIteratorConfig(slice_def=()),
    output_type="dataframe",
    map_axes=True,
)
ra = RegionAnalyzer(racfg)


# Configuration for Cell Segmentation

In [ ]:
# specify preprocessing config for cells
cell_ppcfg = PreprocessFlowConfig(
    processors = [
        #DoGConfig(
        #    sigma_low=1, # 5, 
        #    sigma_high=2, #12, 
        #    normalize=True,
        #    iterator_config=ArrayIteratorConfig(slice_def=(-2,-1)) # YX over each Z focal plane and channel
        #)
    ]
)


In [ ]:
# Specify segmentation config
mscfg = MicroSAMSegmenterConfig(
    iterator_config=ArrayIteratorConfig(slice_def=()),
    embedding_path=embedding_path,
    #gpu_fraction=0.3,
)

min_cell_radius = 2.0
max_cell_radius = 7.0
rfcfg = RegionFilterConfig(
    filters=[
        RangeFilterConfig(
            attribute="cross_sectional_area-xy", 
            range=(np.pi*min_cell_radius**2, np.pi*max_cell_radius**2)
        )
    ]
)

cell_sfcfg = SegmentationFlowConfig(
    segmenter = mscfg,
    region_filter = rfcfg,
)

In [ ]:
@flow
def analyze_cells(labels: list[np.ndarray], metadata: list[dict[str, Any]]) -> list[pd.DataFrame]:
    print ([l.shape for l in labels])
    print ([m["channel_names"] for m in metadata])
    racfg = RegionAnalyzerConfig(
        properties=["volume", "centroid", "cross_sectional_area", "bbox", "aspect_ratio"],
        iterator_config = ArrayIteratorConfig(slice_def=()),
        output_type="dataframe"
    )
    ra = RegionAnalyzer(racfg)

    measurements = ra.run.map(labels, metadata=metadata)

    cdcfg = CoincidenceDetectorConfig(
        method="ios",
        iterator_config=ArrayIteratorConfig(slice_def=()),
        mode="outline",
    )
    label_index_combinations = list(itertools.combinations(range(len(labels)), 2))
    l1 = [labels[c[0]] for c in label_index_combinations]
    l2 = [labels[c[1]] for c in label_index_combinations]
    sn = [(metadata[c[0]]["channel_names"][0], metadata[c[1]]["channel_names"][0]) for c in label_index_combinations]
    print (sn)
    #for la1, la2, sna in zip(l1,l2,sn): 
    cim = CoincidenceDetector(cdcfg).run.map(l1, l2, stack_names=sn)
    return measurements
 

In [ ]:
acfg = AnalysisFlowConfig(
    region_analyzer = RegionAnalyzerConfig(
        properties=["volume", "centroid", "cross_sectional_area", "bbox", "aspect_ratio"],
        iterator_config = ArrayIteratorConfig(slice_def=()),
        output_type="dataframe",
        index_on="object_id",
        map_axes=True,
    ),
    #coincidence_detector = CoincidenceDetectorConfig(
    #    method=IoSMetricsCalculatorConfig(),
    #    iterator_config=ArrayIteratorConfig(slice_def=()),
    #    mode="outline",
    #),
    overlap_calculator = LabelOverlapCalculatorConfig(
        metrics_calculators = [IoSMetricsCalculatorConfig()],
        output_type="dataframe",
        annotate=True,
        triangle=7,
    ),
    overlap_filter = ValueFilterConfig(
        ref_value=0.5,
        axis=0,
        operator=">",
        triangle=LOWER_ND,
        output="masked_values",
    ),
    overlap_aggregator = MatrixAggregatorConfig(
        operation="count",
        axis=1,
    ),
)

In [ ]:
@flow
def full_pipeline(img_path, scene_index=0, outdir=".", embedding_path="embeddings"):
    # load image
    img, metadata = ImageLoader(ilc).run(img_path)
    
    # TISSUE - brain lobes
    # preprocess
    tissue_img, tissue_metadata = PreprocessFlow(tissue_ppcfg).run(img, metadata=metadata, workers=-1)
    tissue_metadata = copy.deepcopy(tissue_metadata)
    tissue_metadata["channel_names"] = ["Lobe"]
    # segment tissue
    tissue_labels = TiledSegmentationFlow(tsfcfg).run(tissue_img, metadata=tissue_metadata, workers=2, verbose=0)

    # BRAIN - all tissue combined
    #tissue_masks = labels_to_masks(tissue_labels)
    brain_label = (tissue_labels>0).astype("uint16")
    brain_metadata = copy.deepcopy(tissue_metadata)
    brain_metadata["channel_names"] = ["Brain"]
    
    # CELLS
    # preprocess
    preprocessed, preprocessed_metadata = PreprocessFlow(cell_ppcfg).run(img, metadata=metadata)
    # split channels
    channels, channel_metadata = unstack_image(preprocessed, preprocessed_metadata, axis=metadata["channel_axis"], strict=False)
    # segment each channel separately
    cell_labels = SegmentationFlow(cell_sfcfg).mapped_run(channels, metadata=channel_metadata)
    
    # Analyze CELLS and TISSUE
    # analyze regions in each channel separately
    combined_labels = [brain_label, tissue_labels, *cell_labels]
    combined_metadata = [brain_metadata, tissue_metadata, *channel_metadata]
    measurements = AnalysisFlow(acfg).run(combined_labels, metadata=combined_metadata)
    # measurements = analyze_cells([*cell_labels, lobe_labels, brain_label], metadata=[*channel_metadata, c_metadata, b_metadata])

    # save labels
    fname_stem = Path(img_path).stem
    imc = ImageWriterConfig(overwrite=True)
    outpaths = [os.path.join(outdir, f'{fname_stem}.scene-{meta.get("scene_index","")}.tif') for meta in combined_metadata]
    # print (outpaths)
    ImageWriter(imc).run.map(combined_labels, outpaths, metadata=combined_metadata)
    
    # make sure to resolve the futures to results
    return (
        resolve_futures(combined_labels),
        resolve_futures(combined_metadata),
        resolve_futures(measurements),
    )


In [ ]:
combined_labels, combined_metadata, measurements = full_pipeline(path, scene_index=scene_index, outdir=".", embedding_path=embedding_path)

In [ ]:
#combined_metadata

In [ ]:
stackview.slice(np.concatenate(combined_labels, axis=-1))

In [ ]:
for k,v in measurements.items():
    print (k, type(v))
df=measurements["region_analyzer_all"].sort_values(["channel","label"])
df

# Query graph for object ancestor lineage

1. Subcellular cellular Dpn -> tissue lobe -> organ brain
2. Count descendants in each channel

In [ ]:
dag = measurements["containment_graph"]

gqcfg = NXGraphQueryConfig(
    attributes=["descendant_counts", "ancestor_lineage"],
    filter_attribute="channel",
    filter_value="Brain",
    include_attributes=["label", "channel"],
    lineage_value_attribute="label",
    output_type="dataframe",
)
gq = NXGraphQuery(gqcfg)
result = gq.run(dag, node=None)
df_counts = gq.format(result["descendant_counts"])
df_lineage = gq.format(result["ancestor_lineage"])
df = pd.concat([df_counts, df_lineage], axis=1)
df = df.loc[:, ~df.columns.duplicated()].sort_values(["label"])
df

In [ ]:
from pyvis.network import Network

net = Network(notebook=True, cdn_resources='in_line', bgcolor="#222222", font_color="white", select_menu=True)
net.barnes_hut()

# Convert the networkx object
net.from_nx(dag)
neighbor_map = net.get_adj_list()

# add neighbor data to node hover data
for node in net.nodes:
    node["title"] = node["object_name"] #+ "\n" +"  Neighbors:\n" + "\n".join(neighbor_map[node["id"]])
    #node["value"] = len(neighbor_map[node["object_name"]])

# Render
net.show("nx_graph.html")

In [ ]:
net.show_buttons(filter_=['physics'])

# Hierarchical label decomposition

# View in Napari

In [ ]:
import napari
viewer = napari.Viewer()

In [ ]:
scale = metadata["physical_pixel_sizes"]
channel_colors = ("green", "blue", "red")
nimg = img#np.expand_dims(img, axis=0)

# add brain label
viewer.add_labels(brain, name="Brain", scale=scale)

# add lobe labels
for ch, l, m in zip(metadata["channel_names"],cell_labels, cell_measurements):
    new_m = m.copy().reset_index()
    background = pd.DataFrame({c: [0] if c=="label" else [np.nan] for c in new_m.columns.to_list()})
    new_m = pd.concat([background, new_m], ignore_index=True)
    # print (new_m)
    viewer.add_labels(l, name=f"{ch}-Labels", features=new_m, scale=scale)

# add cell labels for each channel
ch_images, ch_metadata = unstack_image(img, metadata=metadata, axis="C", strict=False)
for name, c_img, color in zip(metadata["channel_names"], ch_images, channel_colors):
    viewer.add_image(c_img, name=f"{name}", scale=scale, colormap=color, blending="additive")
    


In [ ]:
dl1 = viewer.layers["Dpn-Labels-Lobe 1"]
dl2 = viewer.layers["Dpn-Labels-Lobe 2"]
print (np.intersect1d(np.unique(dl1.data), np.unique(dl2.data)))
